# Medical Students Dataset: Data Preprocessing Walkthrough

**Author:** Faith Njah

This notebook takes a raw, messy health dataset and turns it into a clean, model-ready table. Each step explains *what* was done and *why*.

**Dataset:** [Medical Students Dataset (Kaggle)](https://www.kaggle.com/datasets/slmsshk/medical-students-dataset): demographic and health measurements (age, height, weight, blood type, BMI, temperature, heart rate, blood pressure, cholesterol, diabetes, smoking) for medical students.

**Steps**
1. Load and explore the data
2. Classify the features
3. Handle missing values
4. Remove duplicates
5. Detect and handle outliers
6. Fix data types
7. Feature engineering (BMI)
8. Encode categorical variables
9. Feature scaling
10. Exploratory visualisation
11. Summary

## 1. Setup and load the data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style="whitegrid")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv("../data/medical_students_dataset.csv")
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

## 2. Classify the features

| Column | Type | Classification |
|---|---|---|
| Student ID | Identifier | Not a feature |
| Age | Numerical | Discrete (whole years) |
| Gender | Categorical | Nominal |
| Height (cm) | Numerical | Continuous |
| Weight (kg) | Numerical | Continuous |
| Blood Type | Categorical | Nominal |
| BMI | Numerical | Continuous |
| Temperature | Numerical | Continuous |
| Heart Rate | Numerical | Discrete (beats per minute) |
| Blood Pressure | Numerical | Continuous |
| Cholesterol | Numerical | Continuous |
| Diabetes | Categorical | Nominal (binary Yes/No) |
| Smoking | Categorical | Nominal (binary Yes/No) |

In [ ]:
NUMERIC_COLS = ["Age", "Height", "Weight", "BMI", "Temperature",
                "Heart Rate", "Blood Pressure", "Cholesterol"]
NOMINAL_COLS = ["Gender", "Blood Type"]
BINARY_COLS = ["Diabetes", "Smoking"]

data = df.copy()  # keep the raw data untouched

## 3. Handle missing values

In [ ]:
missing = data.isnull().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(data) * 100).round(1)
missing

In [ ]:
complete_rows = data.dropna().shape[0]
print(f"Rows with no missing values at all: {complete_rows:,} of {len(data):,} "
      f"({complete_rows / len(data):.0%})")

**Observation:** every column has missing values, and they are spread across different rows. Dropping every incomplete row (`dropna()`) would throw away a large share of the data, so instead:

- rows with no **Student ID** are dropped (they can't be tracked or de-duplicated);
- **numeric** columns are filled with the **median** (robust to outliers);
- **categorical** columns are filled with the **mode** (most frequent value).

In [ ]:
data = data.dropna(subset=["Student ID"])

for col in NUMERIC_COLS:
    data[col] = data[col].fillna(data[col].median())
for col in NOMINAL_COLS + BINARY_COLS:
    data[col] = data[col].fillna(data[col].mode().iloc[0])

print("Missing values left:", int(data.isnull().sum().sum()))
print(f"Shape after handling missing data: {data.shape}")

## 4. Remove duplicates

In [ ]:
print("Exact duplicate rows:   ", data.duplicated().sum())
print("Repeated Student IDs:   ", data["Student ID"].duplicated().sum())

data = data.drop_duplicates()
data = data.drop_duplicates(subset=["Student ID"], keep="first").reset_index(drop=True)
print(f"Shape after removing duplicates: {data.shape}")

**Observation:** each student should appear once, so both exact duplicates and repeated IDs are removed, keeping the first record. The result is assigned back to `data`. (The original version called `drop_duplicates()` without saving the result, so nothing was actually removed.)

## 5. Detect and handle outliers

In [ ]:
data[NUMERIC_COLS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, NUMERIC_COLS):
    sns.boxplot(y=data[col], ax=ax, color="#7aa6c2")
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Numeric features: box plots", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "boxplots.png", dpi=120)
plt.show()

Outliers are flagged with the **IQR rule**: a value is an outlier if it falls below Q1 - 1.5×IQR or above Q3 + 1.5×IQR. Values are also sanity-checked against realistic ranges (for example, adult height of roughly 150–200 cm).

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

report = []
for col in NUMERIC_COLS:
    low, high = iqr_bounds(data[col])
    n_out = ((data[col] < low) | (data[col] > high)).sum()
    report.append({"column": col, "lower": round(low, 2), "upper": round(high, 2), "outliers": n_out})
pd.DataFrame(report)

**Decision:** rather than deleting rows (and losing the rest of that student's data), outliers are **capped** at the IQR fences. This keeps the sample size and limits the influence of extreme values.

In [ ]:
for col in NUMERIC_COLS:
    low, high = iqr_bounds(data[col])
    data[col] = data[col].clip(lower=low, upper=high)

## 6. Fix data types

In [ ]:
data["Student ID"] = data["Student ID"].astype(int)
data["Age"] = data["Age"].round().astype(int)
data[["Student ID", "Age"]].dtypes

## 7. Feature engineering

### 7a. Recompute BMI
BMI = weight (kg) / height (m)². Height is recorded in centimetres, so it is converted to metres first. The recomputed value is compared with the BMI column supplied in the dataset.

In [ ]:
data["BMI_Computed"] = (data["Weight"] / (data["Height"] / 100) ** 2).round(2)
data[["Height", "Weight", "BMI", "BMI_Computed"]].head()

In [ ]:
diff = (data["BMI"] - data["BMI_Computed"]).abs()
print(f"Mean absolute difference between given and computed BMI: {diff.mean():.2f}")

**Observation:** if the given BMI and the computed BMI disagree, the given column is not consistent with height and weight. The computed value is the trustworthy one because it comes straight from the definition.

### 7b. BMI category
The computed BMI is grouped into the WHO categories so it can be analysed as an ordered category.

In [ ]:
data["BMI_Category"] = pd.cut(
    data["BMI_Computed"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    right=False,
)
data["BMI_Category"].value_counts()

### 7c. Averages by gender

In [ ]:
data.groupby("Gender")[["Age", "Weight", "Height", "BMI_Computed"]].mean().round(2)

## 8. Exploratory visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

data["Gender"].value_counts().plot.pie(autopct="%1.1f%%", ax=axes[0], colors=["#7aa6c2", "#e8a87c"])
axes[0].set_title("Gender distribution")
axes[0].set_ylabel("")

sns.histplot(data=data, x="Age", hue="Gender", multiple="dodge", discrete=True, ax=axes[1])
axes[1].set_title("Age distribution by gender")

sns.countplot(data=data, x="BMI_Category", hue="Gender", ax=axes[2])
axes[2].set_title("BMI category by gender")

fig.tight_layout()
fig.savefig(FIG_DIR / "distributions.png", dpi=120)
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = data[NUMERIC_COLS + ["BMI_Computed"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Correlation between numeric features")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=120)
plt.show()

## 9. Encode categorical variables

| Column | Method | Why |
|---|---|---|
| Diabetes, Smoking | Binary mapping (No → 0, Yes → 1) | Only two values; one column is enough |
| Gender, Blood Type | One-hot encoding | Nominal: no natural order, so integer labels would imply a false ranking |
| BMI_Category | Ordinal encoding | Has a real order (Underweight < Normal < Overweight < Obese) |

In [ ]:
print("Unique blood types:", sorted(data["Blood Type"].unique()))

encoded = data.copy()
for col in BINARY_COLS:
    encoded[col] = encoded[col].map({"No": 0, "Yes": 1}).astype(int)
encoded["BMI_Category_Code"] = encoded["BMI_Category"].cat.codes
encoded = pd.get_dummies(encoded, columns=NOMINAL_COLS, dtype=int)
encoded.head()

## 10. Feature scaling

Features are on very different scales (age in years, cholesterol in mg/dL, height in cm). Distance-based and gradient-based models are sensitive to this, so both common techniques are compared:

- **Z-score standardisation** → mean 0, standard deviation 1. Best when a feature is roughly normally distributed.
- **Min-max normalisation** → range [0, 1]. Best when the distribution is not normal or a bounded range is needed.

In [ ]:
scale_cols = NUMERIC_COLS + ["BMI_Computed"]

standardised = encoded.copy()
standardised[scale_cols] = StandardScaler().fit_transform(encoded[scale_cols])

normalised = encoded.copy()
normalised[scale_cols] = MinMaxScaler().fit_transform(encoded[scale_cols])

pd.DataFrame({
    "z-score mean": standardised[scale_cols].mean().round(2),
    "z-score std": standardised[scale_cols].std().round(2),
    "min-max min": normalised[scale_cols].min().round(2),
    "min-max max": normalised[scale_cols].max().round(2),
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(encoded["Cholesterol"], ax=axes[0], bins=30)
axes[0].set_title("Cholesterol: original")
sns.histplot(standardised["Cholesterol"], ax=axes[1], bins=30, color="#e8a87c")
axes[1].set_title("Cholesterol: z-score")
sns.histplot(normalised["Cholesterol"], ax=axes[2], bins=30, color="#85c79a")
axes[2].set_title("Cholesterol: min-max")
fig.tight_layout()
fig.savefig(FIG_DIR / "scaling_comparison.png", dpi=120)
plt.show()

**Observation:** scaling changes the units but not the shape of the distribution. The standardised version is kept as the final output; in a real project the choice would be confirmed by comparing model performance on both.

In [ ]:
standardised.to_csv("../data/medical_students_clean.csv", index=False)
print(f"Final dataset: {standardised.shape[0]:,} rows x {standardised.shape[1]} columns")

## 11. Summary

| Step | What was done |
|---|---|
| Missing values | Dropped rows without an ID; median imputation for numeric columns and mode for categorical ones |
| Duplicates | Removed exact duplicates and repeated Student IDs |
| Outliers | Detected with the IQR rule and capped at the fences |
| Types | Student ID and Age converted to integers |
| Feature engineering | Recomputed BMI from height and weight; added WHO BMI categories |
| Encoding | Binary mapping, one-hot and ordinal encoding chosen per column type |
| Scaling | Compared z-score and min-max; kept z-score |

The output, `data/medical_students_clean.csv`, is ready for modelling (for example, predicting diabetes risk from health indicators).